# 6. A Technically Credible MetaHarness Evaluation

This notebook turns an initially ambiguous panel comparison into a controlled applied-AI systems experiment. It is written for AI engineers, forward-deployed engineers, solution architects, and researchers who need to decide whether MemoRizz adds useful memory and orchestration—or merely adds latency and calls.

We will build the harnesses and MemAgents from scratch, state the causal questions before seeing results, inspect the earlier artifact without overstating it, and analyze the completed paid factorial run. The default path is safe: it reads the committed sanitized artifact and performs no external model calls.


## Example result: a bounded Opus 5 regression

This is the committed result produced by the task-native small suite later in this notebook. It is shown up front so you know the shape of the engineering decision before reading the implementation. The source of truth is the [sanitized evaluation artifact](../../eval/results/2026-08-23-metaharness-opus5-small-suite.json); the final section reconstructs this table from that JSON and binds it to the artifact SHA-256.

| Configuration | What it means | Model(s) | Memory | Hidden checks | Fully correct tasks | Total latency | Cost | Tokens | Calls |
|---|---|---|---|---:|---:|---:|---:|---:|---:|
| MemAgent + Codex | One memory-grounded MemAgent sends every task to Codex | GPT-5.6 Luna, medium effort | Filesystem | 31/32 (96.875%) | 3/4 | 268.064 s | $0.041852 | 474,610 | 4 |
| MemAgent + Claude Code | One memory-grounded MemAgent sends every task to Claude Code | Claude Opus 5, medium effort | Filesystem | 31/32 (96.875%) | 3/4 | 190.259 s | $0.592389 | 94,996 | 4 |
| MemoRizz Always-on Panel | Luna repairs first; Opus independently reviews the resulting workspace on every task | GPT-5.6 Luna → Claude Opus 5, medium effort | Filesystem | 31/32 (96.875%) | 3/4 | 610.007 s | $0.950665 | 566,302 | 8 |
| MemoRizz Risk-routed Panel | A preregistered risk rule chooses one harness per task: Luna for ordinary contracts, Opus for security or tenant isolation | GPT-5.6 Luna or Claude Opus 5, medium effort | Filesystem | 32/32 (100%) | 4/4 | 236.718 s | $0.389566 | 215,873 | 4 |

The preregistered cost-quality rule passed for the risk-routed panel in this run. That is a bounded descriptive result, not proof of universal superiority: there are only four synthetic tasks and one execution per task-arm, and the routed Opus call passed one approval check that the separate Opus-only call missed.


## The earlier run was valid—but the broad interpretation was not

The earlier `structured_adaptive_v1` run correctly showed that a MemoRizz panel could start with Codex, verify complete structured coverage, and avoid an unnecessary Claude call. Both panel arms recorded one harness run. That is evidence for **adaptive early stopping**, not evidence that a Codex-plus-Claude panel outperformed either harness.

A technically credible evaluation must keep separate questions separate:

1. **Wrapper overhead:** What changes when the same Codex or Claude call goes through the MemAgent interface?
2. **Coordination value:** Does always running both harnesses improve evidence quality enough to justify its extra calls?
3. **Adaptive value:** How much quality, latency, and cost changes when Claude becomes a coverage-triggered fallback?
4. **Provider sensitivity:** Are differences caused by the orchestration strategy or by Filesystem versus Oracle retrieval?

No single row can answer all four questions.


## Applied use case and decision

Imagine a team deploying a security-review agent for high-impact changes. Codex and Claude Code are both available. The team needs to choose among a direct vendor harness, a memory-governed single harness, an always-on independent panel, and an adaptive panel.

The engineering decision is not simply “which model wins?” It is:

> Which execution strategy lies on the best quality–cost–latency frontier while preserving scope, evidence provenance, host verification, and operational auditability?

The controlled fixture is an export-authorization review with six known requirements. It is useful because correctness, source grounding, structured coverage, verification, calls, tokens, latency, and cost can all be observed. It is still one synthetic task, so it cannot establish general agent superiority.


## Experimental design

```mermaid
flowchart TB
    F[Same fixture, requirements, schema and verification] --> P{Memory provider}
    P --> FS[Filesystem]
    P --> ORA[Oracle AI Database]
    FS & ORA --> S{Execution strategy}
    S --> DC[Direct MetaHarness to Codex]
    S --> MC[MemAgent to Codex]
    S --> DCL[Direct MetaHarness to Claude Code]
    S --> MCL[MemAgent to Claude Code]
    S --> FULL[MemAgent full panel: Codex and Claude]
    S --> ADAPT[MemAgent adaptive panel: Codex, then Claude on gaps]
    DC & MC & DCL & MCL & FULL & ADAPT --> G[Fail-closed validation]
    G --> J[Identity-blind judge plus deterministic metrics]
    J --> E[Paired effects; no general winner claim]
```

With two providers and six strategies, there are 12 arms per repeat. Two counterbalanced repeats produce 24 arm observations. The reverse-order pairing gives every arm the same mean execution position, reducing—but not eliminating—time/order effects.


In [1]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import uuid
from pathlib import Path
from pprint import pprint

from IPython.display import display

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'memorizz').exists():
            return candidate
    raise RuntimeError('Open this notebook from the MemoRizz checkout or a child directory.')

REPO_ROOT = find_repo_root()
for import_root in (REPO_ROOT, REPO_ROOT / 'src'):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from eval.metaharness import factorial_comparison as factorial
from eval.metaharness import provider_comparison as legacy
from memorizz import MemAgentBuilder
from memorizz._env_io import load_layered_env
from memorizz.approval import SQLiteApprovalStore
from memorizz.enums.memory_type import MemoryType
from memorizz.memory_provider.filesystem.provider import FileSystemConfig, FileSystemProvider
from memorizz.metaharness import ClaudeCodeHarness, CodexHarness, MetaHarness, SQLiteHarnessRunStore

loaded_env_files = load_layered_env(extra_paths=[REPO_ROOT / '.env'])
RUN_LIVE = os.getenv('MEMORIZZ_RUN_HARNESS_EVALUATION') == '1'
NEW_RUNNER = REPO_ROOT / 'eval' / 'metaharness' / 'factorial_comparison.py'
LEGACY_ARTIFACT = REPO_ROOT / 'eval' / 'results' / '2026-08-22-metaharness-provider-comparison-optimized.json'
COMMITTED_ARTIFACT = REPO_ROOT / 'eval' / 'results' / '2026-08-22-metaharness-factorial-paid.json'
default_output = (
    Path(tempfile.gettempdir()) / 'memorizz-factorial-comparison.json'
    if RUN_LIVE else COMMITTED_ARTIFACT
)
OUTPUT_PATH = Path(os.getenv('MEMORIZZ_FACTORIAL_EVAL_OUTPUT', str(default_output)))
print({
    'python': Path(sys.executable).name, 'repo': REPO_ROOT.name,
    'env_files_loaded': len(loaded_env_files), 'paid_run_enabled': RUN_LIVE,
})


{'python': 'python', 'repo': 'memorizz', 'env_files_loaded': 2, 'paid_run_enabled': False}


## 1. Freeze the protocol before looking at new results

The protocol manifest records hashes of the exact fixture file bytes, gold answers, prompt, schema, judge rubric, evaluator code, models, provider matrix, randomization seed, cost stop, package version, Git revision, and claim boundaries. The stable protocol fingerprint excludes volatile timestamps, while a separate environment fingerprint captures the runtime. This makes real protocol drift detectable without declaring every rerun a new protocol.

A dry run creates this manifest without initializing Oracle or contacting any model.


In [2]:
manifest = factorial.protocol_manifest(
    repo_root=REPO_ROOT,
    profile='factorial',
    providers=['filesystem', 'oracle'],
    repeats=2,
    seed=20260822,
    codex_model=os.getenv('MEMORIZZ_EVAL_CODEX_MODEL', 'gpt-5.6-luna'),
    claude_model=os.getenv('MEMORIZZ_EVAL_CLAUDE_MODEL', 'sonnet'),
    judge_model=os.getenv('MEMORIZZ_EVAL_JUDGE_MODEL', 'gpt-4.1'),
    maximum_observed_cost_usd=5.0,
)
assert len(manifest['arms']) == 12
assert set(manifest['mean_execution_position'].values()) == {6.5}
assert manifest['paper_comparable'] is False
pprint({
    'protocol': manifest['protocol'],
    'protocol_fingerprint': manifest['protocol_fingerprint'],
    'environment_fingerprint': manifest['environment_fingerprint'],
    'arms_per_repeat': len(manifest['arms']),
    'repeats': manifest['repeats'],
    'mean_execution_position': sorted(set(manifest['mean_execution_position'].values())),
    'paper_comparable': manifest['paper_comparable'],
})
def matrix_model(strategy):
    if strategy in {'direct_codex', 'memagent_codex'}:
        return manifest['models']['codex']
    if strategy in {'direct_claude', 'memagent_claude'}:
        return manifest['models']['claude_code']
    return f"{manifest['models']['codex']} + {manifest['models']['claude_code']}"

matrix_rows = [
    {
        'arm': arm,
        'provider': spec['provider'],
        'strategy': spec['strategy'],
        'model': matrix_model(spec['strategy']),
        'interface': spec['kind'],
        'call_policy': spec['call_policy'],
    }
    for arm, spec in manifest['arms'].items()
]
try:
    import pandas as pd
    display(pd.DataFrame(matrix_rows))
except ImportError:
    pprint(matrix_rows)


{'arms_per_repeat': 12,
 'environment_fingerprint': 'e251b4cc2e4371f4232a01ddc3277521a4bc5fb97ac3e2403f66a0c82d06d63f',
 'mean_execution_position': [6.5],
 'paper_comparable': False,
 'protocol': 'paired_factorial_v1',
 'protocol_fingerprint': '62ca0bca01185018a594020cec9303dea3c6e79fdc9b83b4fbde964a89f67c9b',
 'repeats': 2}


,arm,provider,strategy,model,interface,call_policy
0,direct_codex__filesystem,filesystem,direct_codex,gpt-5.6-luna,direct,exactly_one_codex
1,memagent_codex__filesystem,filesystem,memagent_codex,gpt-5.6-luna,wrapper,exactly_one_codex
2,direct_claude__filesystem,filesystem,direct_claude,sonnet,direct,exactly_one_claude
3,memagent_claude__filesystem,filesystem,memagent_claude,sonnet,wrapper,exactly_one_claude
4,panel_full__filesystem,filesystem,panel_full,gpt-5.6-luna + sonnet,panel,exactly_one_codex_and_one_claude
5,panel_adaptive__filesystem,filesystem,panel_adaptive,gpt-5.6-luna + sonnet,panel,one_codex_then_claude_only_for_missing_coverage
6,direct_codex__oracle,oracle,direct_codex,gpt-5.6-luna,direct,exactly_one_codex
7,memagent_codex__oracle,oracle,memagent_codex,gpt-5.6-luna,wrapper,exactly_one_codex
8,direct_claude__oracle,oracle,direct_claude,sonnet,direct,exactly_one_claude
9,memagent_claude__oracle,oracle,memagent_claude,sonnet,wrapper,exactly_one_claude


## 2. Define the estimands, not just the arms

An estimand is the exact difference we intend to interpret. The evaluator reports paired, left-minus-right deltas:

| Question | Valid comparison | What the delta includes |
|---|---|---|
| MemAgent wrapper overhead | `MemAgent→Codex − Direct MetaHarness→Codex`; `MemAgent→Claude − Direct MetaHarness→Claude` | MemAgent routing, durable run integration, learning evidence, and post-run bookkeeping around the same vendor call |
| Incremental coordination value | `Full panel − MemAgent→Codex`; `Full panel − MemAgent→Claude` | A second independent reviewer plus deterministic structured union, after controlling for the MemAgent layer |
| End-to-end deployment choice | `Full panel − Direct MetaHarness→Codex`; `Full panel − Direct MetaHarness→Claude` | The complete MemoRizz coordination bundle; this is not a pure coordination effect |
| Adaptive-routing value | `Adaptive panel − Full panel` | Savings or losses from skipping Claude when Codex covers every required criterion |
| Provider sensitivity | `Oracle − Filesystem` for the **same strategy** | Retrieval/storage/provider overhead without changing orchestration |

Here **direct** means `MetaHarness.run()` without a MemAgent wrapper. It deliberately retains the same scoped memory context, policy envelope, and verification gate. A raw vendor CLI would remove those controls and confound the wrapper estimate, so raw CLI performance is outside this experiment. We also do not compare an Oracle panel directly with a Filesystem-only baseline and call the difference “MemoRizz value”; that would confound provider and strategy.


## 3. Controls and validity gates

Every arm receives a cold, unique scope containing the same source-linked requirements. Before spend, MemoRizz retrieves every scope and requires one provider-neutral content fingerprint across the full matrix. Every ordinary worker receives the exact same base task, structured schema, read-only workspace, no network, no MCP access, model identifier, per-executed-harness budget, and host verification. Only an actually triggered adaptive fallback receives an additional host-generated list of missing criteria; that prompt change is part of the adaptive treatment.

The evaluator fails closed when execution, schema, root or worker grounding, verification, token/cost telemetry, call policy, or structured coverage is invalid. The full panel must execute exactly one Codex and one Claude run; the adaptive panel must prove Codex is the primary task and execute no more than two runs. The observed-cost stop is checked before each next arm, but it is not a preflight quote because a call can report cost only after it completes. Model aliases such as `sonnet` are recorded but are not immutable snapshots; a paper-grade run must supply and verify dated model identifiers.


## 4. Initialize the real interfaces from scratch

The next cells create the same class of objects used by the paid evaluator. They build a disposable Filesystem provider so this educational path works without Oracle. Construction and probing are read-only and do not execute a vendor model.


In [3]:
DEMO_ROOT = Path(tempfile.mkdtemp(prefix='memorizz-factorial-notebook-'))
WORKSPACE = DEMO_ROOT / 'workspace'
WORKSPACE.mkdir()
legacy._write_fixture(WORKSPACE)
provider = FileSystemProvider(FileSystemConfig(
    root_path=DEMO_ROOT / 'memory',
    embedding_provider=None,
    lazy_vector_indexes=True,
    use_faiss=False,
))
MEMORY_ID = f'notebook-06-{uuid.uuid4().hex[:10]}'
USER_ID = 'tutorial-evaluator'
THREAD_ID = 'topology-only'
source_id = provider.store(
    {
        'title': 'Export policy requirements',
        'content': f'Applicable review requirements:\n{legacy.REQUIREMENTS}',
        'user_id': USER_ID,
        'thread_id': THREAD_ID,
    },
    MemoryType.KNOWLEDGE_BASE,
    memory_id=MEMORY_ID,
)
codex_model = manifest['models']['codex']
claude_model = manifest['models']['claude_code']
service = MetaHarness(
    memory_provider=provider,
    adapters=[CodexHarness(default_model=codex_model), ClaudeCodeHarness(default_model=claude_model)],
    run_store=SQLiteHarnessRunStore(DEMO_ROOT / 'runs.sqlite3'),
    approval_store=SQLiteApprovalStore(DEMO_ROOT / 'approvals.sqlite3'),
    allowed_workspace_roots=[str(WORKSPACE)],
)
probes = {name: service.probe(name) for name in ('codex', 'claude-code')}
for name, probe in probes.items():
    print(f"{name:<12} ready={probe['ready']} available={probe['available']}")
    if not probe['ready']:
        print('  reason:', probe.get('error'))
        print('  remediation:', probe.get('remediation'))


codex        ready=False available=False
  reason: Version probe failed
  remediation: Install 'codex', ensure it is executable, then rerun `memorizz harness doctor codex`.
claude-code  ready=True available=True


## 5. Build wrapper and panel MemAgents

The preceding probe describes the current notebook kernel only. A nested or sandboxed notebook process can fail a vendor CLI version probe even when the host CLI is healthy; the saved paid artifact separately records that Codex 0.149.0 and Claude Code 2.1.202 were both ready before spend. Run `memorizz harness doctor codex` and `memorizz harness doctor claude-code` in the launch shell when preparing a live rerun. The paid cell refuses to run while either current-kernel probe is not ready.

A runtime-backed MemAgent forwards the complete task to a named harness through MemoRizz's governed envelope. This lets us measure direct `MetaHarness.run()` against MemAgent→Codex or MemAgent→Claude without changing the underlying vendor call policy. Both wrapper and panel MemAgents record bounded learning evidence, so the full-panel-minus-wrapper pair does not accidentally compare learning-on with learning-off.

The coordinating roots are model-less. Planning is fixed in code and consolidation is a deterministic structured union, preventing an unreported third model from confounding cost and quality.


In [4]:
shared_config = {
    'workspace': str(WORKSPACE),
    'permissions': {
        'workspace_mode': 'read_only',
        'allowed_roots': [str(WORKSPACE)],
        'network': 'none',
        'mcp_access': 'none',
        'allowed_env': [],
    },
    'verification': {'command': f'"{sys.executable}" -B verify.py', 'timeout_seconds': 30},
    'output_schema': legacy.FINDING_OUTPUT_SCHEMA,
}
codex_agent = (
    MemAgentBuilder().with_name(f'Factorial Codex wrapper {MEMORY_ID}')
    .with_memory_provider(provider).with_memory_ids(MEMORY_ID)
    .with_execution_harness('codex', meta_harness=service, config={
        **shared_config, 'model': codex_model, 'budget': legacy.CODEX_BUDGET.to_dict(),
    })
    .with_learning_control_plane(True, {
        'evidence_token_budget': 900, 'evidence_max_items': 4,
        'compile_async': False, 'compile_every_n_events': 0,
    })
    .with_semantic_cache(enabled=False).as_ephemeral().build(validate=False)
)
claude_agent = (
    MemAgentBuilder().with_name(f'Factorial Claude wrapper {MEMORY_ID}')
    .with_memory_provider(provider).with_memory_ids(MEMORY_ID)
    .with_execution_harness('claude-code', meta_harness=service, config={
        **shared_config, 'model': claude_model, 'budget': legacy.CLAUDE_BUDGET.to_dict(),
    })
    .with_learning_control_plane(True, {
        'evidence_token_budget': 900, 'evidence_max_items': 4,
        'compile_async': False, 'compile_every_n_events': 0,
    })
    .with_semantic_cache(enabled=False).as_ephemeral().build(validate=False)
)

def build_coordinator(name, *, adaptive):
    plan = [
        {
            'task_id': 'policy-review',
            'description': legacy.COMMON_TASK,
            'assigned_agent_id': codex_agent.agent_id,
            'priority': 1, 'dependencies': [],
        },
        {
            'task_id': 'verification-review',
            'description': legacy.COMMON_TASK,
            'assigned_agent_id': claude_agent.agent_id,
            'priority': 1, 'dependencies': [],
        },
    ]
    return (
        MemAgentBuilder().with_name(f'{name} {MEMORY_ID}')
        .with_instruction('Coordinate source-grounded reviews without a synthesis model call.')
        .with_memory_provider(provider).with_memory_ids(MEMORY_ID)
        .with_learning_control_plane(True, {
            'evidence_token_budget': 900, 'evidence_max_items': 4,
            'compile_async': False, 'compile_every_n_events': 0,
        })
        .with_semantic_cache(enabled=False)
        .with_delegation(
            [codex_agent, claude_agent], enabled=True, mode='deterministic',
            plan=plan, return_report=True, persist_participants=False,
            evidence_context=True, consolidation_strategy='structured',
            required_finding_ids=legacy.REQUIRED_FINDING_IDS,
            adaptive_escalation={
                'enabled': adaptive, 'escalation_task_ids': ['verification-review'],
                'criterion_descriptions': legacy.FINDING_CRITERIA,
            },
            max_consolidation_result_chars=8_000,
        )
        .as_ephemeral().build(validate=False)
    )

full_panel = build_coordinator('Full panel', adaptive=False)
adaptive_panel = build_coordinator('Adaptive panel', adaptive=True)


In [5]:
topology = {
    'wrappers': [
        {'mode': codex_agent.meta_harness_mode, 'harness': codex_agent.default_harness, 'native_model': codex_agent.model is not None},
        {'mode': claude_agent.meta_harness_mode, 'harness': claude_agent.default_harness, 'native_model': claude_agent.model is not None},
    ],
    'full_panel': {
        'root_model': full_panel.model,
        'strategy': full_panel.delegation_config['consolidation_strategy'],
        'adaptive': full_panel.delegation_config['adaptive_escalation']['enabled'],
    },
    'adaptive_panel': {
        'root_model': adaptive_panel.model,
        'strategy': adaptive_panel.delegation_config['consolidation_strategy'],
        'adaptive': adaptive_panel.delegation_config['adaptive_escalation']['enabled'],
    },
}
pprint(topology)
assert [agent.default_harness for agent in full_panel.delegates] == ['codex', 'claude-code']
assert full_panel.model is None and adaptive_panel.model is None
assert full_panel.delegation_config['adaptive_escalation']['enabled'] is False
assert adaptive_panel.delegation_config['adaptive_escalation']['enabled'] is True
assert service.list_runs(limit=10) == []


{'adaptive_panel': {'adaptive': True,
                    'root_model': None,
                    'strategy': 'structured'},
 'full_panel': {'adaptive': False,
                'root_model': None,
                'strategy': 'structured'},
 'wrappers': [{'harness': 'codex', 'mode': 'runtime', 'native_model': False},
              {'harness': 'claude-code',
               'mode': 'runtime',
               'native_model': False}]}


## 6. Use memory-first capabilities where they are causally relevant

| Capability | Evaluation policy | Reason |
|---|---|---|
| Tenant-scoped retrieval and provenance | Enabled for every arm | Tests the same grounded evidence path and prevents cross-arm leakage |
| Provider-neutral content fingerprint | Required before spend | Proves Filesystem and Oracle supplied equivalent evidence content |
| Host verification and normalized traces | Required | A plausible response is not operational success |
| Learning-control-plane evidence | Enabled for coordinated panels | Captures workflow evidence without automatically promoting a skill |
| Semantic cache | Disabled for cold comparisons | Cached output would change model-call count and confound the effects |
| Summarization and compaction | Not forced into one turn | Their value appears in long histories; here they add irrelevant overhead |
| Forgetting | Not applied during a run | It is a reviewed lifecycle operation, not a benchmark shortcut |

A separate warm-production profile should measure cache, summary, and compaction savings. Mixing cold scientific comparison with warm production optimization makes both harder to interpret.


## 7. Metrics and judge design

The experiment combines deterministic operational metrics—schema coverage, grounding, verification, exact calls, tokens, actions, latency, and cost basis—with a secondary identity-blind judge for correctness, gold coverage, evidence, actionability, grounding, and unsupported claims. Host verification is an execution-health gate; it does **not** certify that the review answer is correct.

Candidate identities, source UUIDs, and temporary paths are hidden from the judge. Each candidate is scored in a separate stateless call, preventing the judge from anchoring on or directly comparing another arm's answer. Ranking is derived only after independent scores are returned. A single OpenAI judge can still have calibration, style, and provider-family bias, so its score is not objective ground truth. Deterministic gold-ID coverage remains separate, and a broader study should add another independent judge plus human adjudication of a stratified disagreement sample.


## 8. Reinterpret the earlier artifact correctly

The earlier artifact remains useful evidence. We preserve it as the adaptive-routing predecessor and inspect its actual call counts instead of rewriting history.


In [6]:
legacy_artifact = json.loads(LEGACY_ARTIFACT.read_text(encoding='utf-8'))
assert legacy_artifact['status'] == 'valid'
assert legacy_artifact['paper_comparable'] is False
legacy_rows = [
    {
        'arm': item['arm'], 'provider': item['memory_provider'],
        'judge/100': item['judge_score_mean'], 'gold/6': item['gold_findings_mean'],
        'unsupported': item['unsupported_claims_mean'],
        'latency_s': round(item['wall_latency_ms_mean'] / 1_000, 2),
        'cost_usd': item['cost_usd_mean'],
        'harness_calls': item.get('harness_runs_mean', 1.0),
    }
    for item in legacy_artifact['results']
]
try:
    display(pd.DataFrame(legacy_rows))
except NameError:
    pprint(legacy_rows)
assert all(row['harness_calls'] == 1.0 for row in legacy_rows)
print('Conclusion: the panel rows measured adaptive Codex-first execution, not an executed two-harness panel.')


,arm,provider,judge/100,gold/6,unsupported,latency_s,cost_usd,harness_calls
0,codex_only,filesystem,93.5,6.0,0.5,60.45,0.007854,1.0
1,claude_only,filesystem,100.0,6.0,0.0,83.94,0.140066,1.0
2,memorizz_panel_filesystem,filesystem,94.5,6.0,0.0,54.01,0.008035,1.0
3,memorizz_panel_oracle,oracle,93.5,6.0,0.0,64.63,0.007784,1.0


Conclusion: the panel rows measured adaptive Codex-first execution, not an executed two-harness panel.


The defensible conclusion is narrow: on this fixture, structured adaptive routing retained all six findings and avoided the fallback. The old artifact cannot estimate independent Codex-plus-Claude review because Claude did not execute inside either panel arm. The corrected `panel_full` arm closes that gap by requiring and validating both harness calls.


## 9. Estimate spend before opting in

A full two-provider, two-repeat matrix executes approximately 16 Codex calls and 12–16 Claude calls, depending on adaptive escalation, plus 24 independent judge calls (one per arm observation). The estimate below uses the earlier fixture's mean execution costs as a planning range—not a quote.


In [7]:
legacy_by_arm = {item['arm']: item for item in legacy_artifact['results']}
codex_cost = legacy_by_arm['codex_only']['cost_usd_mean']
claude_cost = legacy_by_arm['claude_only']['cost_usd_mean']
codex_calls = 2 * 2 * 4  # providers × repeats × direct/wrapper/full/adaptive
claude_calls_min = 2 * 2 * 3  # direct/wrapper/full
claude_calls_max = 2 * 2 * 4  # adaptive also escalates
execution_low = codex_calls * codex_cost + claude_calls_min * claude_cost
execution_high = codex_calls * codex_cost + claude_calls_max * claude_cost
pprint({
    'codex_calls': codex_calls,
    'claude_calls_range': [claude_calls_min, claude_calls_max],
    'planning_execution_cost_range_usd': [round(execution_low, 2), round(execution_high, 2)],
    'judge_model_calls': 24,
    'judge_cost': 'additional and model/output dependent',
    'configured_observed_execution_stop_usd': 5.0,
})


{'claude_calls_range': [12, 16],
 'codex_calls': 16,
 'configured_observed_execution_stop_usd': 5.0,
 'judge_cost': 'additional and model/output dependent',
 'judge_model_calls': 24,
 'planning_execution_cost_range_usd': [1.81, 2.37]}


## 10. Optional paid factorial run

Configure local Oracle and authenticate both vendor CLIs before starting Jupyter. Credentials belong in the process environment or secure credential stores, never in notebook cells. The runner preflights Oracle, both harnesses, and every evidence scope before its first paid call.

```bash
export MEMORIZZ_RUN_HARNESS_EVALUATION=1
export MEMORIZZ_EVAL_MAX_OBSERVED_COST_USD=5
python -m jupyter lab examples/metaharness
```

For a cheaper mechanics check, use `--profile smoke`: Filesystem, one repeat, and six strategies. It cannot estimate provider sensitivity or repeat stability.


In [8]:
command = [
    sys.executable, str(NEW_RUNNER), '--execute', '--profile', 'factorial',
    '--max-observed-cost-usd', os.getenv('MEMORIZZ_EVAL_MAX_OBSERVED_COST_USD', '5.0'),
    '--output', str(OUTPUT_PATH),
]
if RUN_LIVE:
    not_ready = {name: probe.get('error') for name, probe in probes.items() if not probe.get('ready')}
    if not_ready:
        raise RuntimeError(f'Both external harnesses must be ready before paid execution: {not_ready}')
    subprocess.run(command, cwd=REPO_ROOT, check=True)
    print('Corrected artifact:', OUTPUT_PATH)
else:
    print('Safe default: no external harness or judge was executed.')
    print('Prepared command: python eval/metaharness/factorial_comparison.py --execute --profile factorial --output <configured-output>')


Safe default: no external harness or judge was executed.
Prepared command: python eval/metaharness/factorial_comparison.py --execute --profile factorial --output <configured-output>


## 11. Audit the completed paid artifact

The committed artifact contains 24 arm observations and 28 external harness calls. It is sanitized: credentials and machine-specific path prefixes are excluded. Positive paired deltas mean left minus right. Wrapper pairs isolate the MemAgent interface; full-versus-adaptive pairs isolate mandatory redundancy versus coverage-gated early stopping; provider pairs compare the same strategy.

The audit also preserves the judge failures instead of hiding them. Four initial replies returned rubric maxima as fractions (`0.35`, `0.30`, …) instead of points (`35`, `30`, …). A complete all-candidate rejudge then used construct-irrelevant factors such as verbosity and instruction restatement. A stricter pass failed closed. Required-criterion recall, grounding, and host verification are therefore the primary quality gates; scalar judge scores remain visible but are invalid for comparison. Known spend is a lower bound because four failed judge attempts predated durable failed-call accounting.

The runner sets `winner_allowed=false`: one fixture and two repeats provide descriptive systems evidence, not a statistically general claim.


In [9]:
corrected = None
if OUTPUT_PATH.exists():
    candidate = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
    if candidate.get('schema_version') == factorial.SCHEMA_VERSION:
        corrected = candidate
if corrected and corrected.get('valid'):
    paid_models = corrected['protocol_manifest']['models']
    def paid_model(strategy):
        if strategy in {'direct_codex', 'memagent_codex'}:
            return paid_models['codex']
        if strategy in {'direct_claude', 'memagent_claude'}:
            return paid_models['claude_code']
        return f"{paid_models['codex']} + {paid_models['claude_code']}"
    rows = [{
        'arm': item['arm'],
        'model': paid_model(item['strategy']),
        'criterion_recall': item['required_criterion_recall_mean'],
        'diagnostic_judge_score': item['judge_score_mean'],
        'latency_s': round(item['wall_latency_ms_mean'] / 1000, 3),
        'cost_usd': item['cost_usd_mean'],
        'tokens': item['total_tokens_mean'],
        'calls': item['harness_calls_mean'],
    } for item in corrected['summaries']]
    try:
        display(pd.DataFrame(rows))
    except NameError:
        pprint(rows)
    effects = corrected['paired_effects']
    def operational_delta(payload):
        return {key: payload[key] for key in (
            'required_criterion_recall_mean_delta',
            'wall_latency_ms_mean_delta', 'cost_usd_mean_delta',
            'total_tokens_mean_delta',
        )}
    pprint({
        'cost_accounting': corrected['cost_accounting'],
        'filesystem_adaptive_minus_full': operational_delta(effects['coordination_value']['adaptive_vs_full__filesystem']),
        'oracle_adaptive_minus_full': operational_delta(effects['coordination_value']['adaptive_vs_full__oracle']),
        'judge_scale_corrections': corrected['reporting_corrections']['fractional_judge_scale_record_count'],
        'llm_scalar_valid_for_comparison': corrected['quality_validity']['llm_judge_scalar_valid_for_comparison'],
    })
    assert corrected['validation_failures'] == []
    assert corrected['interpretation_policy']['winner_allowed'] is False
    assert corrected['context_matrix_preflight']['ok'] is True
    assert corrected['quality_validity']['primary_metric_valid'] is True
    assert corrected['quality_validity']['llm_judge_scalar_valid_for_comparison'] is False
    assert all(report['report']['ok'] for report in corrected['cleanup'])
else:
    print('No valid corrected paid artifact is present. The notebook will not manufacture results from the legacy run.')


,arm,model,criterion_recall,diagnostic_judge_score,latency_s,cost_usd,tokens,calls
0,direct_codex__filesystem,gpt-5.6-luna,100.0,96.0,52.226,0.007763,47066,1
1,memagent_codex__filesystem,gpt-5.6-luna,100.0,99.5,49.918,0.006751,60724,1
2,direct_claude__filesystem,sonnet,100.0,100.0,72.796,0.116509,12699,1
3,memagent_claude__filesystem,sonnet,100.0,100.0,87.226,0.136621,14039,1
4,panel_full__filesystem,gpt-5.6-luna + sonnet,100.0,99.5,76.962,0.129007,66937,2
5,panel_adaptive__filesystem,gpt-5.6-luna + sonnet,100.0,96.5,42.882,0.006237,53250,1
6,direct_codex__oracle,gpt-5.6-luna,100.0,98.5,45.396,0.005066,46616,1
7,memagent_codex__oracle,gpt-5.6-luna,100.0,93.5,56.030,0.007203,56465,1
8,direct_claude__oracle,sonnet,100.0,100.0,76.234,0.123828,13181,1
9,memagent_claude__oracle,sonnet,100.0,100.0,67.434,0.109011,12206,1


{'cost_accounting': {'all_paid_call_costs_complete': False,
                     'execution_harness_calls': 28,
                     'execution_observed_usd': 1.65243825,
                     'initial_judge_estimated_usd': 0.122654,
                     'initial_judge_model_calls': 24,
                     'judge_cost_complete': False,
                     'judge_estimated_usd': 0.217664,
                     'judge_model_calls': 53,
                     'known_cost_judge_model_calls': 49,
                     'rejudge_estimated_usd': 0.09501,
                     'rejudge_model_calls': 25,
                     'stop_budget_is_post_call_observed_not_a_preflight_quote': True,
                     'stop_budget_usd': 5.0,
                     'total_measured_usd': 1.87010225,
                     'total_measured_usd_excludes_unknown_failed_rejudge_calls': True,
                     'unaccounted_failed_rejudge_calls': 4,
                     'unknown_execution_cost_records': 0},
 'filesyst

## 12. What would support a broader research claim?

This protocol closes the main causal-design gaps for the controlled fixture, but it does not create external validity. The fixture is the unit of inference, so two reruns of the same fixture are repeated measurements—not `n=2` independent tasks.

| Threat | Remaining risk | Paper-grade mitigation |
|---|---|---|
| Internal validity | Vendor drift, warm provider caches, and nonlinear time effects can survive reverse-order balancing | Pin dated snapshots, record cache telemetry, rotate blocks across days, and rerun failed blocks rather than cherry-picking |
| Construct validity | One LLM judge may reward style or share provider-family bias | Add deterministic task metrics, a second judge family, and blinded human disagreement adjudication |
| External validity | One synthetic security-review fixture represents a narrow task distribution | Preregister several held-out task families and report per-family effects |
| Statistical conclusion | Repeats are correlated pseudo-replicates | Use independent tasks as the sample unit and bootstrap or model task-level paired effects |
| Reproducibility | Floating aliases and CLI releases can change | Record CLI versions, exact model snapshots, code/data hashes, environment lock, and raw sanitized traces |

That larger suite can support claims about classes of agent tasks. This notebook supports a precise claim about one applied systems use case. Honesty about that boundary is part of technical credibility.


## 13. Keep memory retrieval evaluation separate

The latest memory-suite rerun showed 92–94% warm-ingestion reduction and improved AgentMemBench MRR, while LoCoMo-Plus Recall@6 remained zero. Its gold-evidence oracle isolated retrieval as the primary failure on that sample. Those findings guide retrieval work, but they are not MetaHarness coordination evidence.


In [10]:
memory_suite_path = REPO_ROOT / 'eval' / 'results' / '2026-08-22-memory-suite-fixed-rerun.json'
memory_suite = json.loads(memory_suite_path.read_text(encoding='utf-8'))
pprint({
    'schema_version': memory_suite['schema_version'],
    'paper_comparable': memory_suite['paper_comparable'],
    'validation': memory_suite['validation'],
    'efficiency': memory_suite['efficiency'],
    'conclusion': memory_suite['conclusion'],
})
assert memory_suite['paper_comparable'] is False
assert memory_suite['validation']['full']['failed'] == 0


{'conclusion': 'The latest changes materially improved reproducibility, '
               'observability, retrieval rank on the AgentMemBench sample, and '
               'repeated-run ingestion speed. They did not improve LoCoMo-Plus '
               'cognitive recall on the fixed sample, and the one-case '
               'answer-score changes are too small to support an aggregate '
               'quality claim.',
 'efficiency': {'agentmembench_hosted_ingestion_reduction_percent': 92.59,
                'cost_increase_note': 'The new total includes gold-evidence '
                                      'oracle generation and, for judge-scored '
                                      'cases, an additional oracle grading '
                                      'call.',
                'hosted_total_estimated_cost_usd': 0.034405,
                'locomo_plus_hosted_ingestion_reduction_percent': 93.69,
                'previous_hosted_total_estimated_cost_usd': 0.021005},
 'paper_comparable

## 14. Reconstruct and reference the Opus 5 result table

The opening table is convenient for readers; this section makes it auditable. The code reads the committed artifact, creates OPUS5_RESULT_ROWS and OPUS5_RESULT_TABLE, and publishes OPUS5_RESULT_REFERENCE with the relative artifact path, artifact SHA-256, protocol fingerprint, schema version, and generation timestamp.

The primary metric is the micro pass rate over 32 held-out task-native checks. The scalar judge is absent by design. All arms used Filesystem memory, source grounding, durable approval, public verification, medium effort, and fresh workspaces. Semantic-cache reuse and summarization were disabled because these are cold, single-turn repair tasks.


In [11]:
OPUS5_ARTIFACT = REPO_ROOT / 'eval' / 'results' / '2026-08-23-metaharness-opus5-small-suite.json'
opus5_result = json.loads(OPUS5_ARTIFACT.read_text(encoding='utf-8'))
meaning = {
    'memagent_codex': 'One MemAgent routes every task to Codex',
    'memagent_opus': 'One MemAgent routes every task to Claude Code',
    'panel_always': 'Codex repair followed by Opus review on every task',
    'panel_routed': 'One pre-task risk rule selects Luna or Opus',
}
OPUS5_RESULT_ROWS = [{
    'configuration': item['display_name'],
    'meaning': meaning[item['strategy']],
    'model': item['model_configuration'],
    'memory': item['memory_provider'],
    'hidden_checks': f"{item['hidden_checks_passed']}/{item['hidden_checks_total']}",
    'accuracy_pct': item['accuracy_pct'],
    'fully_correct_tasks': f"{item['fully_correct_tasks']}/{item['task_count']}",
    'total_latency_s': round(item['wall_latency_ms_total'] / 1000, 3),
    'cost_usd': item['cost_usd_total'],
    'tokens': item['total_tokens'],
    'calls': item['harness_calls'],
} for item in opus5_result['summaries']]
OPUS5_RESULT_REFERENCE = {
    'artifact': OPUS5_ARTIFACT.relative_to(REPO_ROOT).as_posix(),
    'artifact_sha256': hashlib.sha256(OPUS5_ARTIFACT.read_bytes()).hexdigest(),
    'schema_version': opus5_result['schema_version'],
    'protocol': opus5_result['protocol_manifest']['protocol'],
    'protocol_fingerprint': opus5_result['protocol_manifest']['protocol_fingerprint'],
    'generated_at': opus5_result['generated_at'],
}
try:
    OPUS5_RESULT_TABLE = pd.DataFrame(OPUS5_RESULT_ROWS)
    display(OPUS5_RESULT_TABLE)
except NameError:
    OPUS5_RESULT_TABLE = OPUS5_RESULT_ROWS
    pprint(OPUS5_RESULT_TABLE)
pprint(OPUS5_RESULT_REFERENCE)
pprint(opus5_result['decision'])
assert opus5_result['valid'] is True
assert opus5_result['accounting']['judge_model_calls'] == 0
assert len(OPUS5_RESULT_ROWS) == 4
assert all(row['memory'] == 'filesystem' for row in OPUS5_RESULT_ROWS)


,configuration,meaning,model,memory,hidden_checks,accuracy_pct,fully_correct_tasks,total_latency_s,cost_usd,tokens,calls
0,MemAgent + Codex,One MemAgent routes every task to Codex,Codex: gpt-5.6-luna (medium effort),filesystem,31/32,96.875,3/4,268.064,0.041852,474610,4
1,MemAgent + Claude Code,One MemAgent routes every task to Claude Code,Claude Code: claude-opus-5 (medium effort),filesystem,31/32,96.875,3/4,190.259,0.592389,94996,4
2,MemoRizz Always-on Panel,Codex repair followed by Opus review on every ...,Codex: gpt-5.6-luna then Claude Code: claude-o...,filesystem,31/32,96.875,3/4,610.007,0.950665,566302,8
3,MemoRizz Risk-routed Panel,One pre-task risk rule selects Luna or Opus,Codex: gpt-5.6-luna or Claude Code: claude-opu...,filesystem,32/32,100.000,4/4,236.718,0.389566,215873,4


{'artifact': 'eval/results/2026-08-23-metaharness-opus5-small-suite.json',
 'artifact_sha256': '71ea5ba241e16dce944628b9f6f1475f2436ba24cce35dd765aa585f4e789be9',
 'generated_at': '2026-08-23T01:44:55.927437+00:00',
 'protocol': 'opus5_task_native_small_suite_v1',
 'protocol_fingerprint': '75eb08ae74f9a84a37807749ef5af2fcd14f08d3302c3ddaaa7370be3258620f',
 'schema_version': 'memorizz.metaharness.opus5-small-suite.v1'}
{'interpretation': 'The preregistered MemoRizz frontier rule passed.',
 'panel_cost_quality_frontier_supported': True,
 'quality_floor_pct': 96.875,
 'routed_minus_codex_accuracy_points': 3.125,
 'routed_minus_opus_accuracy_points': 3.125,
 'routed_minus_opus_cost_usd': -0.20282323,
 'routed_minus_opus_latency_ms': 46459}


## Final takeaways

- MemoRizz is a harness when a native MemAgent owns the loop and a meta-harness when it governs other harnesses behind one contract.
- Direct `MetaHarness.run()` versus MemAgent→Codex or MemAgent→Claude measures wrapper overhead while holding memory context, policy, and verification constant.
- The full panel is the only arm that tests mandatory Codex-plus-Claude coordination.
- The adaptive panel tests whether validated structured coverage can safely avoid a fallback.
- Credible evaluation independently scores candidates, reports calls, provenance, failures, cost basis, and claim boundaries, and treats independent tasks—not reruns—as the statistical sample.
- Task-native deterministic gates are primary. LLM scalar judges are secondary diagnostics and must fail closed when they score construct-irrelevant factors.


In [12]:
full_panel.close(close_memory_provider=False)
adaptive_panel.close(close_memory_provider=False)
codex_agent.close(close_memory_provider=False)
claude_agent.close(close_memory_provider=False)
service.close()
provider.close()
shutil.rmtree(DEMO_ROOT, ignore_errors=True)
print('Removed disposable notebook resources.')


Removed disposable notebook resources.
